# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     | Carlos Enrique Nieves Ochoa |
| **Fecha**      | 21 de septiembre de 2026 |
| **Expediente** | 758210 |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

**¿Por qué?**

Usamos `train_test_split` para entrenar con una parte de los datos y evaluar con otra que el modelo no vio durante el ajuste. Así podemos comparar cómo le va con datos conocidos y nuevos, y tener una idea de si podrá generalizar.

**Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?**

Si usamos todos los datos para entrenar, el modelo puede aprovechar más información, pero ya no tendremos datos aparte para comprobar cómo predice observaciones nuevas. Por eso primero dejamos una parte para evaluarlo. Cuando ya elegimos el modelo y lo validamos, podemos volver a ajustarlo con todos los datos disponibles.

El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### **Ejercicio 1**

**Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.**

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

df = pd.read_excel("Motor Trend Car Road Tests.xlsx")
df = df.drop(columns=["model"])

X = df.drop(columns=["mpg"])
y = df["mpg"]

loo = LeaveOneOut()
mse = []

for train, test in loo.split(X):
    modelo = LinearRegression()
    modelo.fit(X.iloc[train], y.iloc[train])
    y_pred = modelo.predict(X.iloc[test])
    mse.append(mean_squared_error(y.iloc[test], y_pred))

print("Modelos entrenados:", len(mse))
print("MSE promedio:", np.mean(mse))
print("Desviación estándar del MSE:", np.std(mse))

Modelos entrenados: 32
MSE promedio: 12.181558006901955
Desviación estándar del MSE: 17.06739987188854


**Interpreta.**

Se entrenaron 32 modelos, dejando un automóvil distinto para prueba en cada vuelta. El MSE promedio fue 12.18 y su desviación estándar fue 17.07, así que el error cambió bastante según el automóvil que quedó fuera. Esto sugiere que, con solo 32 observaciones, algunas predicciones son mucho más difíciles que otras.

In [5]:
from sklearn.linear_model import Lasso

mse_lasso = []

for train, test in loo.split(X):
    modelo = Lasso(alpha=0.01)
    modelo.fit(X.iloc[train], y.iloc[train])
    y_pred = modelo.predict(X.iloc[test])
    mse_lasso.append(mean_squared_error(y.iloc[test], y_pred))

print("Desviación estándar del MSE con Lasso:", np.std(mse_lasso))

Desviación estándar del MSE con Lasso: 14.96961750357886


## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### **Ejercicio 2**
**Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.**

In [6]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

Dataset Shape: (20640, 8) (20640,)
Dataset Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Dataset Target: ['MedHouseVal']


In [7]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=10)
mse = []

for train, test in kf.split(X):
    modelo = LinearRegression()
    modelo.fit(X[train], y[train])
    y_pred = modelo.predict(X[test])
    mse.append(mean_squared_error(y[test], y_pred))

print("Folds evaluados:", len(mse))
print("MSE promedio:", np.mean(mse))
print("Desviación estándar del MSE:", np.std(mse))

Folds evaluados: 10
MSE promedio: 0.5509524296956629
Desviación estándar del MSE: 0.19288582953865185


**Interpreta.**

Se hicieron 10 ajustes, usando cada fold una vez como prueba y los otros nueve para entrenar. El MSE promedio fue 0.551 y su desviación estándar fue 0.193. El error no fue igual en todos los folds, pero este promedio nos da una idea más completa del desempeño del modelo que una sola división entre entrenamiento y prueba.

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3